# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [10]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
openai = OpenAI()

In [12]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

In [13]:
# here is the question; type over this to ask something new

system_prompt = """
You are provided with the link of a website: https://www.qftest.com/doc/manual/en/manual.html.
It contains a manual of application used for automated testing.
You are able to answer questions about using the application based on the manual.
"""
question = """
Please explain how to use a procedure qfs.database.executeSelectStatement in QF-test.
1. List the variables needed to use this procedure.
2. Explain how to return the result of SQL query used in this procedure, which has one column and several rows?
"""

In [17]:
# Get gpt-4o-mini to answer, with streaming
stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        stream=True
    )
response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)

To use the procedure `qfs.database.executeSelectStatement` in QF-test, follow these steps:

### 1. List the Variables Needed
- **Connection String**: A string that contains the database connection information. It typically includes the database type, server address, database name, user credentials, etc.
- **SQL Statement**: The SQL query that you want to execute (e.g., `SELECT * FROM table_name`).
- **Result Variable**: A variable where the result of the executed SQL query will be stored. This variable should be compatible with the structure of the expected result (for example, a data table or list).

### 2. Returning the Result of the SQL Query
To return the result of an SQL query that outputs one column with several rows, you can follow these steps:

- Execute the `qfs.database.executeSelectStatement` procedure using the appropriate connection string and SQL statement.
- Store the result in the designated result variable.
- To retrieve the values, you can iterate over the rows of the result set stored in the result variable. 

Here is a simple outline on how to retrieve the values:
```plaintext
// 1. Declare variables
String connectionString = "Your_Connection_String";
String sqlStatement = "SELECT your_column FROM your_table;";
List<String> resultList;

// 2. Execute the SQL statement
resultList = qfs.database.executeSelectStatement(connectionString, sqlStatement);

// 3. Process the results
for (String value : resultList) {
    print(value); // or store in another variable as needed
}
```
Make sure to handle any exceptions or errors that may occur during the execution of the SQL statement or when processing the results.

In [19]:
# Get Llama 3.2 to answer
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

stream = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        stream=True
    )
response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)

According to the manual of QF-test, `qfs.database.executeSelectStatement` is a procedure that executes a SELECT statement on a database.

**Variables needed:**

1. `select_sql`: The SQL query string.
2. `[optional]` `$timeout`: The maximum amount of time in milliseconds it can take for the execution to complete (default 5000).
3. `[optional]` `$output_file`: The file where the output will be saved (can be used if an error occurs).

**Usage and returning result:**

1. Call `qfs.database.executeSelectStatement` procedure with your SQL query string as the first argument.
2. If the procedure is successful, it will return a table containing one column (the results of the SQL query) and multiple rows.

To access the returned table, you can use the `$q` object to get its columns:

```qfm
qfs.database.executeSelectStatement("SELECT * FROM my_table WHERE condition");

$columns =
[
  "$col1" : "Column_1",
];

$table = $q->getTable(); // gets the result of qfs.database.executeSelectStatement

if ($table->hasRecordByIndex(0)) {
  record = array();
  record[] = "Row Data"; // first column
  print(record[0]); // prints the first row data
}
```

In your case, since you need to access a column with multiple values:

```qfm
qfs.database.executeSelectStatement("SELECT my_column FROM my_table");

// assuming the variable 'my_row_data' holds the data of one row,
$q->getFirstCol() // gets the first column (data type: str)
my_row_data = $q->getValue(); // or $q->asList();
```

These commands allow you to iterate over and access each row in the result.

However, the best approach will depend on your concrete use case.